# Setup és adatútvonalak

Ebben a részben definiáljuk az adatkönyvtárakat és fájlokat, amelyekből a
RAG pipeline dolgozni fog. A cél, hogy minden útvonal jól reprodukálható legyen.

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline
from langgraph.graph import StateGraph, END
from typing import Dict, Any
from abc import ABC, abstractmethod

DATA_DIR = "../data"
PDF_PATH = f"{DATA_DIR}/1706.03762v7-2.pdf"
PERSIST_DIR = "../chroma_store"

# PDF beolvasás és feldolgozás

Itt olvassuk be a PDF dokumentumot, majd daraboljuk fel kisebb szövegrészekre
(chunkokra). Ez szükséges ahhoz, hogy a későbbi keresés hatékony legyen.

In [ ]:
# PDF reading
reader = PdfReader(PDF_PATH)
pages = [page.extract_text() or "" for page in reader.pages]
full_text = "\n".join(pages)

print("The whole text length", len(full_text))

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

chunks = splitter.split_text(full_text)
print("Number of chunks:", len(chunks))
print("First chunks first 300 character\n", chunks[0][:300])

# Modularitás és cserélhetőség

A pipeline moduláris felépítését mutatjuk be. Az LLM és az embedding modell
absztrakciós osztályok mögé kerül, így könnyen cserélhető. Például:
- Flan-T5 → DummyLLM → más HuggingFace modell
- MiniLM → más SentenceTransformer modell

In [ ]:
#Abstract base embedder class
class BaseEmbedder(ABC):
    @abstractmethod
    def encode(self, texts: list[str]):
        pass


class MiniLM_embedder(BaseEmbedder):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)

    def encode(self, texts: list[str]):
        return self.model.encode(texts, convert_to_numpy=True)

    
# Abstract base class
class BaseLLM(ABC):
    @abstractmethod
    def generate(self, prompt: str, max_new_tokens: int = 256) -> str:
        pass

class HFLLM(BaseLLM):
    def __init__(self, model_name="google/flan-t5-large", device_map="auto"):
        self.pipe = pipeline("text2text-generation", model=model_name, device_map=device_map)

    def generate(self, prompt: str, max_new_tokens: int = 256) -> str:
        out = self.pipe(prompt, max_new_tokens=max_new_tokens)
        return out[0]["generated_text"]

# Dummy fallback model (Purely for demo)
class DummyLLM(BaseLLM):
    def generate(self, prompt: str, max_new_tokens: int = 256) -> str:
        return "This is a dummy answer."

# Embeddingek és ChromaDB tárolás

Ebben a lépésben a chunkokat embeddingekre alakítjuk, majd feltöltjük egy
ChromaDB collection-be. Így a rendszer képes lesz releváns szövegrészeket
visszakeresni. Valamint példányosítun egy llm osztályt.

In [ ]:
# Loading embedding model
embed_model = MiniLM_embedder()

# New chroma client(persistent mode to save the indexes)
client = chromadb.PersistentClient(path=PERSIST_DIR)

# Creating new collection(If exists delete the previous one)
collection_name = "rag_docs"
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(name=collection_name)

# Creating embeddings from chunks
embeddings = embed_model.encode(chunks)

# Uploading documents to collection
ids = [f"doc_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_id": i, "source": PDF_PATH} for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    metadatas=metadatas,
    embeddings=embeddings,
)

print("ChromaDB collection created, documents uploaded:", len(chunks))

# Instantiate Flan-T5-Large modell
Flan_T5_llm = HFLLM()

# Retriever

A retriever feladata, hogy a felhasználói kérdéshez legrelevánsabb chunkokat
keresse ki a ChromaDB-ből. A találatokhoz távolsági score is tartozik, amely
segít a relevancia mérésében.

In [ ]:
def retrieve(query: str, k: int = 3, threshold: float = 0.8):
    q_emb = embed_model.encode([query])
    res = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    docs = []
    for i in range(len(res["documents"][0])):
        score = res["distances"][0][i] 
        
        if score < threshold:  
            docs.append({
                "text": res["documents"][0][i],
                "metadata": res["metadatas"][0][i],
                "score": score
            })
    return docs

# test
results = retrieve("How many layers are there in a encoder?")

for i, doc in enumerate(results):
    print(f"\n--- Result: {i+1} ---")
    print("Score:", doc["score"])
    print("Chunk ID:", doc["metadata"]["chunk_id"])
    print("Text preview:", doc["text"][:300])

# Válasz generálás

Ebben a lépésben a retriever által visszaadott szövegrészekből állítjuk össze
a promptot, majd az LLM generálja a választ. A cél: rövid, pontos válasz a
felhasználói kérdésre.

In [ ]:
def answer_question(query: str, k: int = 1):
    hits = retrieve(query, k=k)
    context_texts = [h["text"] for h in hits]
    joined_context = "\n\n---\n\n".join(context_texts)

    prompt = (
    "Answer the following user query in exactly two sentences. "
    "Use the given context only.\n\n"
    f"User query:\n{query}\n\n"
    f"Context:\n{joined_context}\n\n"
)
    return Flan_T5_llm.generate(prompt)

# Agentic pipeline (LangGraph)

A LangGraph segítségével agentic működést demonstrálunk:
- Planner node: eldönti, kell-e retrieval
- Searcher node: releváns chunkokat keres
- Responder node: választ generál

Ez mutatja az autonóm döntéshozatalt és a részfeladatok önálló végrehajtását.

In [ ]:
State = Dict[str, Any]

# Nodes
def planner_node(state: State) -> State:
    query = state["query"]
    state["needs_retrieval"] = len(query) > 10
    return state

def searcher_node(state: State) -> State:
    if state.get("needs_retrieval"):
        hits = retrieve(state["query"], k=1)
        state["contexts"] = hits
    else:
        state["contexts"] = []
    return state

def responder_node(state: State) -> State:
    contexts = state.get("contexts", [])
    if contexts:  # If there are relevant answers
        joined_context = "\n\n---\n\n".join([c["text"] for c in contexts])
        prompt = (
            "Answer the following question based on the given context in at least two sentences. "
            "If the context is insufficient, say you don't know.\n\n"
            f"Question:\n{state['query']}\n\n"
            f"Context:\n{joined_context}\n\n"
            "Answer:"
        )
    else: 
        prompt = (
            "Answer the following question based on your own knowledge. "
            "Be clear and concise.\n\n"
            f"Question:\n{state['query']}\n\n"
            "Answer:"
        )
    state["answer"] = Flan_T5_llm.generate(prompt, max_new_tokens=256)
    return state


# Graph
graph = StateGraph(State)

graph = StateGraph(State)
graph.add_node("planner", planner_node)
graph.add_node("searcher", searcher_node)
graph.add_node("responder", responder_node)

graph.add_edge("planner", "searcher")
graph.add_edge("searcher", "responder")
graph.add_edge("responder", END)
graph.set_entry_point("planner")

# Compile
app = graph.compile()

# Agentic pipeline (LangGraph)

A LangGraph segítségével agentic működést demonstrálunk:
- Planner node: eldönti, kell-e retrieval
- Searcher node: releváns chunkokat keres
- Responder node: választ generál

Ez mutatja az autonóm döntéshozatalt és a részfeladatok önálló végrehajtását.

In [ ]:
queries = [
    "How many layers are there in an encoder?",
    "What is the role of positional encoding?",
    "Who is the best F1 driver"  # Do not exist in the provided pdf -> LLM fallback
]

for q in queries:
    final_state = app.invoke({"query": q})
    print("\n==========================")
    print("Question:", q)
    print("Answer:\n", final_state["answer"])

# Teljesítmény és bottleneck elemzés

A jelenlegi RAG prototípus működőképes, de vannak szűk keresztmetszetek:

- **PDF feldolgozás**: nagy dokumentumok esetén a beolvasás és chunkolás lassú lehet.
- **Embedding számítás**: CPU-n futtatva a SentenceTransformer modell korlátozott sebességet biztosít.
- **Tárolás és keresés**: ChromaDB jól működik kisebb adathalmazokon, de nagyobb méret esetén optimalizálásra szorulhat.
- **LLM válaszminőség**: a Flan-T5-Large modell gyors és nyílt forrású, de nem mindig ad pontos vagy konzisztens válaszokat.
- **Agentic logika**: jelenleg egyszerű (planner → searcher → responder), de nem tartalmaz önellenőrzést vagy újrakeresést.

Ezek a tényezők határozzák meg a prototípus skálázhatóságát és megbízhatóságát.

# Teljesítménymérés

A prototípus teljesítményét több szempontból lehet értékelni:

- **Retrieval pontosság**: mennyire releváns chunkokat ad vissza a retriever (precision/recall).
- **Latency**: mennyi idő telik el a kérdés beérkezésétől a válasz megjelenéséig (retrieval + LLM).
- **Válaszminőség**: manuális értékelés vagy automatikus metrikák (BLEU, ROUGE) alapján.
- **Skálázhatóság**: hogyan változik a teljesítmény nagyobb dokumentumok vagy több adatforrás esetén.
- **Robusztusság**: hogyan kezeli a rendszer az irreleváns vagy PDF-ben nem szereplő kérdéseket (LLM fallback).

A mérési eredmények segítenek azonosítani, hol érdemes optimalizálni a pipeline-t.

# Továbbfejlesztési lehetőségek

A jelenlegi prototípus demonstrálja az agentic RAG működés alapjait, de számos
irányban továbbfejleszthető:

- **Több dokumentum kezelése**: egyszerre több PDF vagy dataset integrálása,
  hogy a chatbot szélesebb tudásbázissal dolgozzon.
- **Query reformulálás**: agent node, amely újrafogalmazza a felhasználói kérdést,
  ha az első retrieval nem adott releváns találatot.
- **Validator/critic node**: önellenőrző komponens, amely eldönti, hogy a válasz
  elég jó-e, vagy új keresés szükséges.
- **Skálázhatóság**: GPU-optimalizált embedding és LLM modellek használata nagyobb
  teljesítmény érdekében.
- **Teljesítménymérés automatizálása**: metrikák (precision, recall, latency) beépítése
  a pipeline-ba, hogy a rendszer folyamatosan monitorozható legyen.
- **Felhasználói interakciók**: egyszerű UI vagy API endpoint, amelyen keresztül
  a prototípus könnyen kipróbálható.

Ezek a fejlesztések segítenék a prototípusból egy robusztusabb, skálázhatóbb
alkalmazás kialakítását.